<a href="https://colab.research.google.com/github/Abhijith2005binu/AI-ML/blob/main/Uniform_Cost_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
graphs = [
    # Graph 0: Original Tree-like Graph
    {
        'name': "Graph 1: Standard Tree-like Graph",
        'data': {
            'A': [('B', 1), ('C', 4)],
            'B': [('A', 1), ('D', 2), ('E', 5)],
            'C': [('A', 4), ('F', 1)],
            'D': [('B', 2), ('G', 4)],
            'E': [('B', 5), ('G', 1)],
            'F': [('C', 1), ('G', 3)],
            'G': [('D', 4), ('E', 1), ('F', 3)]
        },
        'start': 'A',
        'goal': 'G'
    },

    # Graph 1: 3x3 Grid Network (Grid Map with varying edge costs)
    {
        'name': "Graph 2: 3x3 Grid Layout",
        'data': {
            'N1': [('N2', 2), ('N4', 1)],
            'N2': [('N1', 2), ('N3', 3), ('N5', 1)],
            'N3': [('N2', 3), ('N6', 5)],
            'N4': [('N1', 1), ('N5', 2), ('N7', 6)],
            'N5': [('N2', 1), ('N4', 2), ('N6', 1), ('N8', 3)],
            'N6': [('N3', 5), ('N5', 1), ('N9', 2)],
            'N7': [('N4', 6), ('N8', 2)],
            'N8': [('N5', 3), ('N7', 2), ('N9', 1)],
            'N9': [('N6', 2), ('N8', 1)]
        },
        'start': 'N1',
        'goal': 'N9'
    },

    # Graph 2: Dense Multi-Path Network (Complex connections with bottlenecks)
    {
        'name': "Graph 3: Dense Multi-Path Network",
        'data': {
            'S': [('A', 2), ('B', 5), ('C', 1)],
            'A': [('S', 2), ('D', 3), ('E', 7)],
            'B': [('S', 5), ('E', 1), ('F', 4)],
            'C': [('S', 1), ('F', 6)],
            'D': [('A', 3), ('T', 6)],
            'E': [('A', 7), ('B', 1), ('T', 2)],
            'F': [('B', 4), ('C', 6), ('T', 3)],
            'T': [('D', 6), ('E', 2), ('F', 3)]
        },
        'start': 'S',
        'goal': 'T'
    }
]

# -----------------------------
# 2. Uniform Cost Search Tracking
# -----------------------------
def animate_ucs(graph, start, goal):
    priority_queue = [(0, start, [start])]
    visited = {}
    history = []  # Stores (current_node, visited_nodes_dict, current_path) at each step

    final_path = None
    final_cost = float("inf")

    while priority_queue:
        cost, current_node, path = heapq.heappop(priority_queue)

        if current_node in visited and visited[current_node] <= cost:
            continue

        visited[current_node] = cost
        history.append((current_node, dict(visited), list(path)))

        if current_node == goal:
            final_path = path
            final_cost = cost
            break

        for neighbor, edge_cost in graph[current_node]:
            new_cost = cost + edge_cost
            if neighbor not in visited or new_cost < visited[neighbor]:
                heapq.heappush(priority_queue, (new_cost, neighbor, path + [neighbor]))

    return history, final_path, final_cost

# -----------------------------
# 3. Visualization Setup
# -----------------------------
def visualize_exploration(graph_info, history):
    graph = graph_info['data']
    graph_title = graph_info['name']

    G = nx.Graph()
    for node in graph:
        for neighbor, weight in graph[node]:
            G.add_edge(node, neighbor, weight=weight)

    pos = nx.spring_layout(G, seed=42)
    fig, ax = plt.subplots(figsize=(9, 6))

    def update(frame):
        ax.clear()
        current_node, visited, current_path = history[frame]

        # Determine node colors
        node_colors = []
        for node in G.nodes():
            if node == current_node:
                node_colors.append("gold")       # Currently expanding node
            elif node in current_path:
                node_colors.append("orange")     # Active path
            elif node in visited:
                node_colors.append("lightgreen") # Evaluated/visited nodes
            else:
                node_colors.append("lightblue")  # Unvisited nodes

        # Active path edges
        path_edges = list(zip(current_path, current_path[1:]))

        # Draw Graph
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=850, ax=ax)
        nx.draw_networkx_edges(G, pos, width=2, edge_color="gray", ax=ax)

        if path_edges:
            nx.draw_networkx_edges(G, pos, edgelist=path_edges, width=4, edge_color="red", ax=ax)

        nx.draw_networkx_labels(G, pos, font_size=11, font_weight="bold", ax=ax)
        edge_labels = nx.get_edge_attributes(G, 'weight')
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax)

        ax.set_title(
            f"[{graph_title}]\nStep {frame + 1}/{len(history)}: Current Node = '{current_node}' | Path = {' -> '.join(current_path)}",
            fontsize=11
        )
        ax.axis("off")

    anim = FuncAnimation(fig, update, frames=len(history), interval=1200, repeat=False)
    plt.close()
    return anim

# -----------------------------
# 4. Select Graph & Run
# -----------------------------
# Change index to 0, 1, or 2 to view different graphs
selected_graph_index = 2

selected_graph = graphs[selected_graph_index]
history, path, cost = animate_ucs(selected_graph['data'], selected_graph['start'], selected_graph['goal'])

print(f"--- {selected_graph['name']} ---")
print(f"Start Node   : {selected_graph['start']}")
print(f"Goal Node    : {selected_graph['goal']}")
print(f"Shortest Path: {' -> '.join(path)}")
print(f"Total Cost   : {cost}\n")

# Render animation controls
anim = visualize_exploration(selected_graph, history)
HTML(anim.to_jshtml())

--- Graph 3: Dense Multi-Path Network ---
Start Node   : S
Goal Node    : T
Shortest Path: S -> B -> E -> T
Total Cost   : 8

